In [2]:
import sys

assert sys.version_info >= (3, 10)

In [3]:
from packaging.version import Version
import torch

assert Version(torch.__version__) >= Version("2.6.0")

In [4]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [5]:
import matplotlib.pyplot as plt

plt.rc("font", size=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("legend", fontsize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

In [6]:
import deepxde as dde
import numpy as np

torch.manual_seed(42)
torch.cuda.manual_seed(42)

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Setting up the backend

In [7]:
dde.config.set_default_float("float64")
print(f"Backend: {dde.backend.backend_name}")

Set the default float type to float64
Backend: pytorch


Defining Exact solution

In [8]:
def exact_solution(x):
    return (x + 1) ** 2

Defing a domain Geometry with a boudary of -1 and 1

In [9]:
geom = dde.geometry.Interval(-1, 1)


Defining the Partial Differential Equation here second order so hessian

In [10]:
def pde(x, y):
    dy_xx = dde.grad.hessian(y, x, i=0, j=0)
    return dy_xx

creating a left boundary condition

In [11]:
def boundary_left(x, on_boundary):
    return on_boundary and np.isclose(x[0], -1)

bc_left = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_left)

Creating a right boundary condition

In [12]:
def boundary_right(x, on_boundary):
    return on_boundary and np.isclose(x[0], 1)

bc_right = dde.icbc.NeumannBC(geom, lambda x: 4, boundary_right)

Generating the data using Numpy

In [13]:
observe_x = np.linspace(-1, 1, 35).reshape(-1, 1)
observe_y  = exact_solution(observe_x)
noise  = 0.1 * np.random.randn(35, 1)
observe_y = observe_y + noise
observe = dde.icbc.PointSetBC(observe_x, observe_y, component=0)

Combining all the Data

In [14]:
data  = dde.data.PDE(
    geom, pde,
    [bc_left, bc_right, observe],
    num_domain=3000,
    num_boundary=200,
    num_test=500
)

Building a Neural Network

In [15]:
net  = dde.nn.FNN([1, 256, 128, 64, 1], "tanh", "Glorot uniform")
model = dde.Model(data, net)

Creating a class for generating Self Adaptive Weights through a Callback function during training

In [16]:
class SoftmaxAdaptiveWeights(dde.callbacks.Callback):
    def __init__(self, model, every_n_epochs=100, alpha_0=1.0, gamma=0.999):
        super().__init__()
        self.model = model
        self.every_n_epochs = every_n_epochs
        self.alpha = alpha_0
        self.gamma= gamma
        self.epoch =0

    def on_epoch_end(self):
        self.epoch += 1
        if self.epoch % self.every_n_epochs != 0:
            return

        if len(self.model.losshistory.loss_train) ==0:
            return
        current_losses = self.model.losshistory.loss_train[-1]
        exp_terms = np.exp(self.alpha * np.array(current_losses))
        new_weights = exp_terms / np.sum(exp_terms)
        self.alpha *= self.gamma
        self.model.loss_weights = new_weights.tolist()
        print(f"Epoch {self.epoch}: Adaptive weights updated to {new_weights.tolist()}, alpha decayed to {self.alpha:.4f}")

adaptive_callback = SoftmaxAdaptiveWeights(model, every_n_epochs = 20)

Training the MOdel with 2 stage optimization
1. Adam Optimizer
2. L-BFGS

In [17]:
print("\nStage 1: Adam Optimizer")
model.compile("adam", lr = 0.01,
              loss_weights=[1.0] * 4,
              decay=("inverse time", 1000, 0.1))

losshistory, train_state = model.train(iterations=2000, display_every=200, callbacks=[adaptive_callback])
dde.optimizers.config.set_LBFGS_options(maxiter=200)
model.compile("L-BFGS", loss_weights=model.loss_weights)
losshistory, train_state = model.train(display_every=50,callbacks=[adaptive_callback])


Stage 1: Adam Optimizer
Compiling model...
'compile' took 2.115228 s

Training model...



/home/ziaur/ziazh/lib/python3.14/site-packages/torch/autograd/graph.py:882: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:370.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Step      Train loss                                  Test loss                                   Test metric
0         [9.35e-05, 5.97e-03, 1.54e+01, 3.15e+00]    [8.44e-05, 5.97e-03, 1.54e+01, 3.15e+00]    []  
Epoch 20: Adaptive weights updated to [1.9905546785448573e-07, 2.0022769669151385e-07, 0.9999949480209103, 4.652695925166855e-06], alpha decayed to 0.9990
Epoch 40: Adaptive weights updated to [2.0215063400738456e-07, 2.0333989618980948e-07, 0.9999948843356328, 4.710173837169851e-06], alpha decayed to 0.9980
Epoch 60: Adaptive weights updated to [2.052907599711104e-07, 2.064972843921712e-07, 0.9999948199086921, 4.7683032634447866e-06], alpha decayed to 0.9970
Epoch 80: Adaptive weights updated to [2.0847644968939186e-07, 2.0970046804711773e-07, 0.9999947547322149, 4.827090867401436e-06], alpha decayed to 0.9960
Epoch 100: Adaptive weights updated to [2.1170831450340646e-07, 2.1295006133130658e-07, 0.9999946887982518, 4.886543372377983e-06], alpha decayed to 0.9950
Epoch 120: A

Evaluating the results using Metrics

Create the test points using Numpy

In [18]:
x_test =  np.linspace(-1, 1, 15).reshape(-1, 1)
y_exact = exact_solution(x_test)
y_pred = model.predict(x_test)
noise = 0.1 * np.random.randn(15, 1)
y_test = y_exact + noise


Calculating the performance Metrices

In [19]:
absolute_error = np.abs(y_test  - y_pred)
relative_error = absolute_error / (np.abs(y_test) + 1e-10)
l2_error = np.linalg.norm(y_test - y_pred) / np.linalg.norm(y_test)

u_at_minus1 = model.predict(np.array([[-1.0]]))[0, 0]
u_at_plus1 = model.predict(np.array([[1.0]]))[0, 0]

x_right = np.array([[1.0]])
du_dx_at_1 = model.predict(
    x_right,
    operator=lambda x, y: dde.grad.jacobian(y, x, i=0,j=0)
)[0, 0]

print("📊 PERFORMANCE METRICS")
print(f"L2 Relative Error:      {l2_error:.6f}")
print(f"Max Absolute Error:     {np.max(absolute_error):.6f}")
print(f"Mean Absolute Error:    {np.mean(absolute_error):.6f}")

print("\n📍 BOUNDARY CONDITION CHECK")
print(f"u(-1) = {u_at_minus1:.6f}  (Target: 0.0)")
print(f"du/dx(1) = {du_dx_at_1:.6f}  (Target: 4.0)")



📊 PERFORMANCE METRICS
L2 Relative Error:      0.258157
Max Absolute Error:     0.754548
Mean Absolute Error:    0.441414

📍 BOUNDARY CONDITION CHECK
u(-1) = -0.450777  (Target: 0.0)
du/dx(1) = 3.403224  (Target: 4.0)


Extract the Individual Losses

DeepXDE already computes the exact losses used during training



In [ ]:
final_train_losses = model.losshistory.loss_train[-1]
L_pde, L_bc_l, L_bc_r, L_data = final_train_losses
print("Final training losses (Unweighted - REAL values used by DeepXDE)")
print(f"L_PDE       : {L_pde:.6e}")
print(f"L_BC_left   : {L_bc_l:.6e}")
print(f"L_BC_Right  : {L_bc_r: .6e}")
print(f"L_data      : {L_data: .6e}")
current_weights = model.loss_weights  
weighted = [w * l for w, l in zip(current_weights, final_train_losses)]
print(f"λ_PDE  × L_PDE      : {weighted[0]:.6e}")
print(f"λ_BC_l × L_BC_left  : {weighted[1]:.6e}")
print(f"λ_BC_r × L_BC_right : {weighted[2]:.6e}")
print(f"λ_data × L_data     : {weighted[3]:.6e}")
print(f"\nTotal Weighted Loss : {sum(weighted):.6e}")
print("="*60)

Final training losses (Unweighted - REAL values used by DeepXDE)
L_PDE       : 1.703375e-01
L_BC_left   : 4.920446e-02
L_BC_Right  :  8.670266e-02
L_data      :  7.474413e-02
